# Cómo protegemos un identificador con HMAC-SHA256

Esta notebook reproduce paso a paso el ejemplo de la documentación de `CMAT-research`. El objetivo es mostrar qué entra al algoritmo, qué sale de él y por qué el resultado permite reconocer a la misma persona entre tablas sin conservar su identificador institucional directo.

**Importante:** la clave de esta notebook es ficticia y existe únicamente para la demostración. La clave real nunca debe escribirse en Git, notebooks, documentación o archivos de resultados.

## 1. La idea general

Partimos de un identificador, por ejemplo `175199`. El proceso puede resumirse como:

`ID + tipo de identificador + clave secreta → HMAC-SHA256 → pseudónimo`

Con la misma clave, el mismo ID y el mismo tipo de identificador se obtiene siempre el mismo pseudónimo, lo cual permite enlazar registros longitudinales.

In [1]:
import hashlib
import hmac
import random
import secrets

## 2. Los datos del ejemplo

Usaremos exactamente el mismo ID y la misma clave ficticia de los documentos LaTeX. El `namespace` indica qué tipo de entidad estamos procesando; para estudiantes el repositorio usa `stu`.

In [2]:
student_id = "175199"
namespace = "stu"
demo_key = "7b41e7a1c76f6b50f58a0c628a3f0e8daed7ad4f6dd6c7e97ad31e8f8ad1e933"

print("ID:", student_id)
print("Namespace:", namespace)
print("Longitud de la clave en caracteres:", len(demo_key))

ID: 175199
Namespace: stu
Longitud de la clave en caracteres: 64


La clave tiene **64 caracteres hexadecimales**. Cuando se genera con `secrets.token_hex(32)`, esos caracteres representan 32 bytes aleatorios, es decir, **256 bits de entropía**. La longitud importa porque hace inmenso el espacio de claves posibles, pero la seguridad depende también de que la clave sea verdaderamente aleatoria y permanezca secreta.

## 3. Construimos el mensaje

La función del repositorio no procesa únicamente `175199`; construye `stu:175199`. Incluir `stu` separa criptográficamente los dominios: un valor numérico igual usado como ID de estudiante y como ID de profesor no se trata automáticamente como la misma entidad.

In [3]:
message = f"{namespace}:{student_id}"
print(message)

stu:175199


## 4. Aplicamos HMAC-SHA256

HMAC combina el mensaje con una clave secreta mediante SHA-256. No es un cifrado reversible: no existe una operación de “descifrar” el resultado para recuperar directamente `175199`.

La clave cambia radicalmente la seguridad frente a usar un hash simple. Si solo se publicara `SHA256(ID)`, alguien podría enumerar matrículas plausibles, calcular sus hashes y compararlos. Con HMAC, esa comprobación requiere además conocer la clave secreta.

In [4]:
full_digest = hmac.new(
    demo_key.encode("utf-8"),
    message.encode("utf-8"),
    hashlib.sha256,
).hexdigest()

print(full_digest)

c504a778e1e4b331721d7c88f22c65e0008707cede372864628d04a20bbe27fb


El resultado completo tiene 64 caracteres hexadecimales. `CMAT-research` conserva los primeros 32 caracteres y antepone el espacio de nombres.

In [5]:
pseudonym = f"{namespace}_{full_digest[:32]}"
expected = "stu_c504a778e1e4b331721d7c88f22c65e0"

print(pseudonym)
print("Verificación correcta:", pseudonym == expected)
assert pseudonym == expected

stu_c504a778e1e4b331721d7c88f22c65e0
Verificación correcta: True


El resultado es exactamente el mismo de los documentos:

`175199 → stu_c504a778e1e4b331721d7c88f22c65e0`

Si el mismo estudiante aparece después en otra tabla y se usa la misma clave y el mismo `namespace`, se obtiene la misma etiqueta.

In [6]:
def hmac_pseudonym(value: object, key: str, namespace: str, length: int = 32) -> str:
    message = f"{namespace}:{str(value).strip()}".encode("utf-8")
    digest = hmac.new(key.encode("utf-8"), message, hashlib.sha256).hexdigest()
    return f"{namespace}_{digest[:length]}"

first = hmac_pseudonym(175199, demo_key, "stu")
second = hmac_pseudonym(175199, demo_key, "stu")
print(first)
print(second)
print("Son iguales:", first == second)

stu_c504a778e1e4b331721d7c88f22c65e0
stu_c504a778e1e4b331721d7c88f22c65e0
Son iguales: True


## 5. Generador reproducible con semilla

Para una demostración puede ser útil generar siempre la misma clave a partir de una semilla. La siguiente función hace exactamente eso. **Solo sirve para ejemplos, clases o pruebas**, porque una semilla conocida o predecible permite reconstruir la clave.

Por esa razón, este generador no debe usarse para la clave real de los datos.

In [7]:
def generate_seeded_demo_key(seed: int | str) -> str:
    rng = random.Random(seed)
    return rng.getrandbits(256).to_bytes(32, "big").hex()

seed = "CMAT-demo-2026"
seeded_key = generate_seeded_demo_key(seed)
print("Semilla:", seed)
print("Clave reproducible de demostración:", seeded_key)
print("Caracteres:", len(seeded_key))

Semilla: CMAT-demo-2026
Clave reproducible de demostración: a9ad623d83fe9ed822da1443ce3033d390bbf71815d2bf23934aab8a18db6893
Caracteres: 64


La clave generada en esta sección es deliberadamente distinta de la clave ficticia del ejemplo principal. El ejemplo principal mantiene la misma clave de los documentos LaTeX para que `175199` produzca exactamente el mismo pseudónimo en todos los materiales.

## 6. Cómo se genera una clave real

Para una clave real no buscamos reproducibilidad, sino aleatoriedad criptográfica. Python proporciona `secrets.token_hex(32)`, que genera 32 bytes aleatorios y los representa como 64 caracteres hexadecimales.

La celda queda comentada para evitar que una ejecución rutinaria imprima y guarde accidentalmente una nueva clave en el historial de la notebook. Una clave real debe almacenarse fuera del repositorio.

In [8]:
def generate_secure_key() -> str:
    return secrets.token_hex(32)

# Descomente únicamente para observar el formato de una nueva clave.
# secure_key = generate_secure_key()
# print(secure_key)
# print("Caracteres:", len(secure_key))

## 7. Qué protege este procedimiento y qué no

HMAC-SHA256 protege el **identificador directo** y dificulta los ataques de enumeración que serían posibles con un hash público simple. Sin embargo, los datos siguen siendo pseudonimizados, no anónimos en sentido absoluto, porque variables como fecha, carrera, semestre, materia, tema, calificaciones o una trayectoria longitudinal poco común pueden aportar pistas adicionales.

Por eso la protección completa combina HMAC-SHA256, separación de la clave, control de acceso y minimización de datos.

## 8. Resumen

- ID original: `175199`
- mensaje procesado: `stu:175199`
- algoritmo: HMAC-SHA256
- clave: la misma clave ficticia de los documentos LaTeX
- resultado completo: `c504a778e1e4b331721d7c88f22c65e0008707cede372864628d04a20bbe27fb`
- pseudónimo guardado: `stu_c504a778e1e4b331721d7c88f22c65e0`

El mismo ID, con la misma clave y el mismo espacio de nombres, produce siempre el mismo pseudónimo.